# TP1 : Reed-Solomon codes and star product
## 1. Star product

The following function ``StarProduct`` takes two lists of the same length $u$ et $v$ as inputs and returns the list $u \star v$, whose $i^{th}$ coordinate is the product $u_iv_i$.

In [ ]:
def StarProduct(u,v):
    n=len(u)
    if len(v) != n:
        raise Exception("Cannot compute the star product: vectors do not have the same length.")
    return [u[i]*v[i] for i in range(n)]

**Q1.** Using the function above, write a function ``SquareCode`` with input a code $C$ and returns its star square $C \star C$.
Benefit from the fact that $\star$ is commutative to avoid useless computations.

*To handle to create a code with Sagemaths from a generator matrix ``G``, you can use the command ``codes.LinearCode(G)``.
Actually, if $G$ does not has full rank, this command returns the code generated by the rows of $G$.*

In [ ]:
def SquareCode(C):
    G=C.basis()
    #print(G)
    n = len(G)
    Gresult = []
    for i in range (n):
        for j in range(n):
            vector= StarProduct(G[i], G[j])
            Gresult.append(vector)
    Gresult = Matrix(Gresult)
    CstarC=codes.LinearCode(Gresult)
    return CstarC


#Test
"""
G=random_matrix(GF(2), 3, 4, algorithm='echelonizable', rank =3)
C=codes.random_linear_code(GF(2), 4, 3)
CstarC=SquareCode(C)
print (CstarC.basis())
"""



"\nG=random_matrix(GF(2), 3, 4, algorithm='echelonizable', rank =3)\nC=codes.random_linear_code(GF(2), 4, 3)\nCstarC=SquareCode(C)\nprint (CstarC.basis())\n"

**Q2.** Write a function ``TestRandomSquare(q,n,k,N)`` which picks at random $N$ random linear codes over $\mathbb{F}_q$ of length $n$ and dimension $k$ and returns the list (or the multiset) of the dimension of the Schur square of these $N$ codes.

To build random codes, use ``random_matrix(F, k, n, algorithm='echelonizable', rank=r)`` which returns a random matrix over a finite field ``F`` of size $k \times n$.

In [ ]:
def TestRandomSquare(q,n,k,N):
    ListeDeCodes=[codes.random_linear_code(GF(q), n, k) for i in range (N)]
    L=[]
    for code in ListeDeCodes:
        CstarC=SquareCode(code)
        L.append(CstarC.dimension())
    return(L)

print(TestRandomSquare(2, 4, 3, 10))
        

[3, 4, 3, 3, 4, 4, 3, 3, 3, 3]


**Q3.** Test the function ``TestRandomSquare(n,k,N)`` for $(n,k)=(10,2), (10,4), (20,5), (20,11), (100,11)$ over several finite fields and increasing values of $N$. in the light of the 1st exercise session, what do you observe?

In [ ]:
q=5
N=15

print(float(2*2-2*1/2))
print(TestRandomSquare(q, 10, 2, N))
print(float(4*4-4*3/2))
print(TestRandomSquare(q, 10, 4, N))
print(float(5*5-5*4/2))
print(TestRandomSquare(q, 20, 5, N))
print(float(11*11-11*10/2))
print(TestRandomSquare(q, 20, 11, N))
print(float(11*11-11*10/2))
print(TestRandomSquare(q, 100, 11, N))

#On observe qu'on atteint quasiment tout le temps la borne sauf dans le cas du 20, 11 car on est dans un espace de dimension 20 de base

3.0
[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
10.0
[9, 9, 9, 10, 10, 10, 9, 9, 10, 9, 10, 10, 9, 10, 9]
15.0
[15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15]
66.0
[20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20]
66.0
[66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66, 66]


On observe qu'on atteint quasiment tout le temps la borne sauf dans le cas du 20, 11 car on est dans un espace de dimension 20 de base donc la dimension du code ne peut pas atteindre la borne de la l'exercice 5 qui est plus haute.

## 2. Star square of subcodes of Reed-Solomon codes

In this section, we will notice an unexpected behaviour of subcodes of GRS codes. Let us first work on an example.

### Example

Let $C=  v \star \text{ev}_{\alpha} (\text{Span}(f_1,f_2,f_3,f_4))$ the subcode of dimension $4$ of $\mathsf{GRS}_5(\alpha,v)$ where
* $f_1(X)=1$
* $f_2(X)=X+X^2$
* $f_3(X)=X^3$
* $f_4(X)=X^4$

We will work with $q > n \geq 10$.

**Q4.a.** Construct the code $C$ for a random multiplier $v$ and random evaluation points $\alpha_0,\dots,\alpha_{n-1}$. 

In [ ]:
q=17
n=10
Fq=GF(q)

FqX.<X>=PolynomialRing(Fq)
f1=FqX(1)
f2=X+X**2
f3=X**3
f4=X**4


alpha=[]
for i in range(n):
    rand=Fq.random_element()
    while(rand in alpha):
        rand=Fq.random_element()
    alpha.append(rand)
print(f"alpha={alpha}\n")

v=[]
for i in range(n):
    rand=Fq(0)
    while(rand == Fq(0)):
        rand=Fq.random_element()
    v.append(rand)
print(f"v={v}\n")

def ev(alpha, f):
    L=[]
    for i in range(len(alpha)):
        L.append(f(alpha[i]))
    return vector(L)

def ExternSquareProduct(v, C):
    G=C.basis()
    #print(G)
    n = len(G)
    Gresult = []
    for i in range (n):
        vec= StarProduct(G[i], v)
        Gresult.append(vec)
    Gresult = Matrix(Gresult)
    CstarC=codes.LinearCode(Gresult)
    return CstarC

G=[]
G.append(ev(alpha, f1))
G.append(ev(alpha, f2))
G.append(ev(alpha, f3))
G.append(ev(alpha, f4))
G=Matrix(Fq, G)
C=codes.LinearCode(G)
print(C.basis())
C=ExternSquareProduct(v,C)
print(C.basis())


alpha=[10, 9, 16, 0, 14, 7, 6, 13, 3, 8]

v=[4, 5, 2, 12, 8, 13, 8, 4, 9, 1]

[
(1, 1, 1, 1, 1, 1, 1, 1, 1, 1),
(8, 5, 0, 0, 6, 5, 8, 12, 12, 4),
(14, 15, 16, 0, 7, 3, 12, 4, 10, 2),
(4, 16, 1, 0, 13, 4, 4, 1, 13, 16)
]
[
(4, 5, 2, 12, 8, 13, 8, 4, 9, 1),
(15, 8, 0, 0, 14, 14, 13, 14, 6, 4),
(5, 7, 15, 0, 5, 5, 11, 16, 5, 2),
(16, 12, 2, 0, 2, 1, 15, 4, 15, 16)
]


**Q4.b.** For sanity check, check that $v$ lies in $C$ and that $C$ is a subcode of $\mathsf{GRS}_5(\alpha,v)$.

*N.B.: Use ``codes.GeneralizedReedSolomonCode(list_alpha, k, list_v)`` to construct the GRS code $\mathsf{GRS}_5(\alpha,v)$.*

In [ ]:
v=vector(v)
print(v in C)
GRStest=codes.GeneralizedReedSolomonCode(alpha, 5, v)
for b in C.basis():
    print(b in GRStest)



True
True
True
True
True


**Q5.a.** Check that the square of $C$ coincides with the square of $\mathsf{GRS}_5(\alpha,v)$ using the function ``SquareCode``.
In your tests, take $10 \leq n \leq q$.

In [ ]:
GRStestSquared=SquareCode(GRStest)
CSquared=SquareCode(C)

def EqualCode(C, Cprime):
    for b in C.basis():
        if b not in Cprime:
            return False
    for b in Cprime.basis():
        if b not in C:
            return False
    return True

print(EqualCode(GRStestSquared, CSquared))

True


**Q5.b.** Set $g= f_1f_2-{f_ 2}^2 + f_1f_4+2f_ 1 f_3$. Check that $g(X)=X$ using Sage.
Explain how this proves that $C^{(2)} = \mathsf{GRS}_9(\vec{\alpha},\vec{v} \star \vec{v})$ for any support $\alpha$ and mutiplier $v$ **without using a computer**.

In [ ]:
g = f1*f2 -f2**2 + f1*f4 + 2*f1*f3
print(g(X)==X)

True


On sait déjà que $C\subseteq \mathsf{GRS}_5(\vec{\alpha}, \star \vec{v})$.

Alors $C^2 \subseteq \mathsf{GRS}_5(\vec{\alpha} \star \vec{v}) \star \mathsf{GRS}_5(\vec{\alpha}, \star \vec{v}) = \mathsf{GRS}_9(\vec{\alpha},\vec{v} \star \vec{v})$ (avec la dernière égalité prouvée Exercice 5 question 6).

Or $C=  v \star \text{ev}_{\alpha} (\text{Span}(f_1,f_2,f_3,f_4))$ donc 
$
\begin{align}
   C^2 & =   v \star v \star \text{ev}_{\alpha} (\text{Span}(f_1,f_2,f_3,f_4)) \star \text{ev}_{\alpha} (\text{Span}(f_1,f_2,f_3,f_4)) \\
               & =^{\text{hom}} v \star v \star \text{ev}_{\alpha} (\text{Span}(f_1,f_2,f_3,f_4) \star \text{Span}(f_1,f_2,f_3,f_4))
\end{align}$ 

Et on a que $X = g \in \text{Span}(f_1,f_2,f_3,f_4)) \star (\text{Span}(f_1,f_2,f_3,f_4))$ par définition de g comme combinaison de produit de $f_i$.

De même $X^2=f_2*f_1 - g$ et on peut retrouver pareillement les $1, X^3, X^4, X^5, X^6, X^7, X^8$.

Donc $v \star v \star \text{ev}_{\alpha} \text{Span}(X^i | 0\le 8) \subseteq C^2$.

Or par définition $v \star v \star \text{ev}_{\alpha} \text{Span}(X^i | 0\le 8) = {GRS}_9(\vec{\alpha},\vec{v} \star \vec{v})$.

On a bien montré l'égalité.


### Random subcodes

**Q6.** Write a function ``RandomSubGRS(q,n,k,m)`` which returns a random GRS code ``C`` of dimension $k$ (over $\mathbb{F}_q$ for random distinct $\alpha_0, \ldots, \alpha_{n-1}$ and random non-zero multipliers $v_0,\dots,v_{n-1}$) and a random $m$-dimensional subcode of ``C``. 

*N.B.: Given a generator $k\times n$ matrix $G$ of a code $C$, subcodes of $C$ are those whose generator matrix are of the form $MG$ with a matrix $M$ of size $m \times k$ of full rank.*

In [ ]:
def RandomSubGRS(q,n,k,m):
    Fq=GF(q)
    FqX.<X>=PolynomialRing(Fq)

    alpha=[]
    for i in range(n):
        rand=Fq.random_element()
        while(rand in alpha):
            rand=Fq.random_element()
        alpha.append(rand)

    v=[]
    for i in range(n):
        rand=Fq(0)
        while(rand == Fq(0)):
            rand=Fq.random_element()
        v.append(rand)

    GRSrand=codes.GeneralizedReedSolomonCode(alpha, k, v)
    M=random_matrix(Fq, m, k, algorithm='echelonizable', rank=m)
    G=Matrix(GRSrand.basis())

    return GRSrand, codes.LinearCode(M*G)


"""
#test
q=17
n=10
k=5
m=3
print(RandomSubGRS(q,n,k,m))
"""

'\n#test\nq=17\nn=10\nk=5\nm=3\nprint(RandomSubGRS(q,n,k,m))\n'

**Q7.** Using the previous functions, write a function ``SquareSubGRS(q,n,k,m,N)`` which picks $N$ times a random subcode ``Csub`` of random GRS codes ``C`` and counts how many times the Schur squares of ``C`` and ``Csub`` coincide.

In [ ]:
def SquareSubGRS(q,n,k,m,N):
    count=0
    for _ in range(N):
        C, Csub = RandomSubGRS(q,n,k,m)
        CstarC=SquareCode(C)
        #print(dimension(CstarC))
        CsubstarCsub=SquareCode(Csub)
        #print(dimension(CsubstarCsub))
        if EqualCode(CstarC, CsubstarCsub):
            count+=1
    return count

**Q8.a.** Check that ``SquareSubGRS(q=25,n=9,k=6,m=3,N=200)`` and ``SquareSubGRS(q=32,n=9,k=5,m=3,N=200)`` return $0$. Explain the results.

In [ ]:
print(SquareSubGRS(q=25,n=9,k=6,m=3,N=200))
print(SquareSubGRS(q=32,n=9,k=5,m=3,N=200))



0
0


- 1er cas. 

On a que la dimension du GRSsquare devrait être 11 or 11>9 donc la dimention est 9

Le sous groupe est borné par une dimension $m*m - \frac{m(m-1)}{2} = 6$. Donc par dimension c'est impossibles qu'ils coincident.

- 2ème cas.
  
On a que la dimension du GRSsquare devrait être 9 ce qui est encore plus grand que la borne de dimension du sous groupe.

On peut décommenter les prints d'au dessus pour s'en convaincre.


**Q8.b.** Compute ``SquareSubGRS(q,n,k,m,N=200)`` for several values of $q,n,k,m$ satisfying $2k \leq \frac{m(m+1)}{2}$. Comment your results.

In [ ]:

def Test_SchurSquare_2():
    N=200
    q=[next_prime(2^i) for i in range(5)]
    n=[i for i in range(5, 10)]
    m=[i for i in range(5, 11)]

    for i in range(5):
        for j in range(5):
            for l in range(5):
                k=floor(m[i]*(m[i]+1)/4)
                count = SquareSubGRS(q[j],n[l],k,m[i],N)
                if count>0:
                    print(f"On a {count} collisions pour n={n[l]}, k={k}, m={m[i]}, q = {q[j]}\n")


#Test_SchurSquare_2() ultra long quoi que je fasse 


def Test_SchurSquare_2_bis():
    N=200
    q=[next_prime(2^i) for i in range(5)]
    n=[5 for i in range(5)]
    m=[i for i in range(4)]
    i=randint(0,3)
    j=randint(0,4)
    l=randint(0,4)
    k=floor(m[i]*(m[i]+1)/4)
    count = SquareSubGRS(q[j],n[l],k,m[i],N)
    print(count)

#Test_SchurSquare_2_bis() ultra long quoi que je fasse






KeyboardInterrupt: 

En fait le GRSsquared sera de dimension $min(n, 2k-1)$
Et le C sera borné par $m^2 - \frac{m(m+1)}{2} \le m^2 - 2k$. 

Donc pour avoir une égalité il faudrait que $2k-1<m^2-2k$
ce qui équivaut à $m>\sqrt{4k-1}$, or on a déjà $4k \leq m(m+1)$ d'où 

Ou bien $n<m^2-2k$
Soit $m>\sqrt{n+2K}$